# Project 4 - Personal Library Assistant

#### Objective: Combine all three kinds of memory: short-term thread state, long-term cross-thread memory, and per-run runtime context.

##### Test
```
User alice (premium), thread 1: "I love sci-fi and can't stand horror. Remember that."
User alice, NEW thread 2: "Recommend me something to read."
User bob (free), thread 3: "Recommend me something to read."
```

##### Expected Output
```
[interrupt] refund(expense_id='E-102', amount=820.0) -> you choose: edit amount to 600
[resumed] Refunded 600 for E-102. Approved E-103 (no approval needed, under 500).
[employee] I can only list expenses; refunds need a manager.
```

#### Step 1: Import Library and Environment

In [57]:
# Import Libraries
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent, AgentState
from dotenv import load_dotenv
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from dataclasses import dataclass
from typing import Literal
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
import json, operator
from pprint import pprint
from typing import Annotated

# Load Environmental Files
load_dotenv(override = True)

True

#### Step 2: Read Catalouge JSON File

In [58]:
with open('catalouge.json', 'r') as file: 
    catalouge = json.load(file)
pprint(catalouge)

{'fantasy': [{'author': 'Patrick Rothfuss', 'title': 'The Name of the Wind'},
             {'author': 'Ursula K. Le Guin', 'title': 'A Wizard of Earthsea'},
             {'author': 'Brandon Sanderson',
              'title': 'Mistborn: The Final Empire'},
             {'author': 'J.R.R. Tolkien', 'title': 'The Hobbit'},
             {'author': 'Susanna Clarke', 'title': 'Piranesi'},
             {'author': 'N.K. Jemisin', 'title': 'The Fifth Season'}],
 'horror': [{'author': 'Stephen King', 'title': 'The Shining'},
            {'author': 'Shirley Jackson',
             'title': 'The Haunting of Hill House'},
            {'author': 'Silvia Moreno-Garcia', 'title': 'Mexican Gothic'},
            {'author': 'Josh Malerman', 'title': 'Bird Box'},
            {'author': 'William Peter Blatty', 'title': 'The Exorcist'},
            {'author': 'Mark Z. Danielewski', 'title': 'House of Leaves'}],
 'mystery': [{'author': 'Arthur Conan Doyle',
              'title': 'The Hound of the Baskerville

In [59]:
# Define a runtime context dataclass Context with user_id: str and tier: Literal['free', 'premium'].
@dataclass
class Context:
    user_id: str
    tier: Literal['free', 'premium']

# Define custom state that extends AgentState with books_discussed: list[str].
class CustomState(AgentState):
    books_discussed: Annotated[list[str], operator.add]

In [60]:
@tool
def save_preference(key: str, value: str, runtime: ToolRuntime[Context, CustomState]) -> str:
    """Save the user preference described in key, value pairs at runtime store."""
    user_id = runtime.context.user_id
    runtime.store.put((user_id, "preferences"), key, {'value': value})
    return 'User preferences updated successfully'

@tool
def get_preferences(runtime: ToolRuntime[Context, CustomState]) -> str:
    """Fetch user preferences from store. If user preferences is not found, return not found statement."""
    user_id = runtime.context.user_id
    items = runtime.store.search((user_id, "preferences"))
    return " | ".join(f"{i.key}: {i.value}" for i in items) if items else "No preferences saved."

@tool
def recommend_books(genre: str, runtime: ToolRuntime[Context, CustomState]) -> list[str]:
    """Returns at most 2 titles for free users and 5 for premium based on the genre. If genre not found in catalouge,
    default it to popular and return the titles from popular"""
    tier = runtime.context.tier
    limit = 5 if tier == "premium" else 2
    if genre in catalouge:
        return [b['title'] for b in catalouge[genre][:limit]]
    else:
        return [b['title'] for b in catalouge['popular'][:limit]]

@tool
def log_book(title: str, runtime: ToolRuntime[Context, CustomState]) -> Command:
    """Record a book title after recommending or discussing it."""
    return Command(update={
        "books_discussed": [title],
        "messages": [ToolMessage(f"Logged {title}.", tool_call_id=runtime.tool_call_id)],
    })

In [76]:
SYSTEM_PROMPT = (
    "You are a library assistant. "
    "If the user states a like or dislike, call save_preference. "
    "Before recommending, call get_preferences and never recommend genres the user dislikes. "
    "After recommending, call log_book once for each title you recommend."
)

store = InMemoryStore()

# Create Agent With State-Context Schema, Store, And Checkpointer.
agent = create_agent(model = 'gpt-5-nano', 
                     tools = [save_preference, get_preferences, recommend_books, log_book],
                     state_schema = CustomState,
                     context_schema = Context,
                     store = store,
                     system_prompt = SYSTEM_PROMPT,
                     checkpointer = InMemorySaver())

In [77]:
thread_A = {"configurable": {"thread_id": "1"}}

In [78]:
response = agent.invoke(
    {'messages': [HumanMessage("I love sci-fi and can't stand horror. Remember that.")]},
    context=Context(user_id="Alice", tier='premium'),    
    config = thread_A
)
print(response['messages'][-1].content)

Here are sci-fi picks for you, avoiding horror as requested. I’ve also logged these titles to your reading list.

- Dune — Epic desert saga with politics, religion, and power struggles.
- The Left Hand of Darkness — Classic exploration of gender and society in a distant world.
- Neuromancer — Groundbreaking cyberpunk tale of AI and cyberspace.
- Project Hail Mary — Space survival adventure with smart science and a hopeful twist.
- The Three-Body Problem — Hard SF with alien contact and complex physics.

Would you like more recommendations, or want to narrow by subgenre (space opera, cyberpunk, hard SF), length, or era? I can fine-tune the picks.


In [79]:
thread_B = {"configurable": {"thread_id": "2"}}
response = agent.invoke(
    {'messages': [HumanMessage("Recommend me something to read.")]},
    context=Context(user_id="Alice", tier='premium'),    
    config = thread_B
)
print(response['messages'][-1].content)

Here are sci-fi picks you might enjoy (since you like sci-fi and dislike horror):

- Dune — An epic, politics-forward saga set on the desert planet Arrakis.
- The Left Hand of Darkness — A thought-provoking exploration of gender and society on an alien world.
- Neuromancer — A cyberpunk classic about AI, cyberspace, and corporate intrigue.
- Project Hail Mary — A space survival adventure with grounded science and a dash of humor.
- The Three-Body Problem — A dense, mind-bending hard sci‑fi about alien contact and civilization-scale stakes.

Would you like a quick summary of any title, or should I tailor more options (e.g., shorter/longer reads, newer releases), or pull a few similar titles?


In [80]:
thread_C = {"configurable": {"thread_id": "3"}}
response = agent.invoke(
    {'messages': [HumanMessage("Recommend me something to read.")]},
    context=Context(user_id="Bob", tier='free'),    
    config = thread_C
)
print(response['messages'][-1].content)

Here are a couple of recommendations you might enjoy:
- Project Hail Mary
- The Thursday Murder Club

If you want more options or specific vibes (genre, tone, length), tell me your preferences and I’ll tailor more picks.


In [81]:
snapshotA = agent.get_state(thread_A)
print(snapshotA.values['books_discussed'])

['Dune', 'The Left Hand of Darkness', 'Neuromancer', 'Project Hail Mary', 'The Three-Body Problem']


In [82]:
snapshotB = agent.get_state(thread_B)
print(snapshotB.values['books_discussed'])

['Dune', 'The Left Hand of Darkness', 'Neuromancer', 'Project Hail Mary', 'The Three-Body Problem']


In [83]:
snapshotC = agent.get_state(thread_C)
print(snapshotC.values['books_discussed'])

['Project Hail Mary', 'The Thursday Murder Club']


In [85]:
alice_prefs = store.search(("Alice", "preferences"))
bob_prefs = store.search(("Bob", "preferences"))

assert len(alice_prefs) > 0
assert len(bob_prefs) == 0

bob_reply = response["messages"][-1].content
for title in ["Dune", "Neuromancer", "The Left Hand of Darkness", "The Three-Body Problem"]:
    assert title not in bob_reply, f"Bob saw {title}"

print("Isolation Test passed")

Isolation Test passed
